In [ ]:
import jax.numpy as jnp

from popsim.simulators.comet_mirror.build_default import build_default
from popsim.simulators.comet_mirror.integrator import Integrator


# Test that the simulator runs.
model, state, params = build_default()
sim = Integrator(model)
ts = jnp.linspace(0, 2, 100)

print(params)
sol, debugs = sim(ts, state, params, debug_info=True)

In [ ]:
import jax
import xarray as xr
import hvplot.xarray  # This import is necessary to use hvplot with xarray objects


def keypath_to_string(keypath):
    # Convert a keypath to a string.
    # For some reason, there is often a leading dot in the keypath.
    return ".".join(str(x).lstrip(".") for x in keypath)


def solution_to_xarray(sol):
    # Convert a solution to an xarray.
    leaves_with_path = jax.tree_util.tree_leaves_with_path(sol.ys)
    variables = {keypath_to_string(keypath): leaf for keypath, leaf in leaves_with_path}
    variables = {key: ("time", value) for key, value in variables.items()}
    time = sol.ts
    dataset = xr.Dataset(data_vars=variables, coords={"time": time})
    return dataset


dataset = solution_to_xarray(sol)
dataset.hvplot.line(x="time", by="variable", width=700, height=400, subplots=True).cols(
    1
)